In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter
from pathlib import Path
from IPython.display import display, Markdown

import re as _re

PLOT_DIR = Path('embedding_output/plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

def savefig(fig, name, dpi=150):
    """Save figure to PLOT_DIR and display a markdown link instead of inline image."""
    path = PLOT_DIR / name
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.close(fig)
    display(Markdown(f'**Plot saved:** [{name}]({path})'))

_APPROP_RE = _re.compile(r'\s*This flow is considered \w+\.\s*$')

def strip_appropriateness(text: str) -> str:
    """Remove the trailing 'This flow is considered {appropriate/inappropriate}.' from flow text."""
    if not isinstance(text, str):
        return text
    return _APPROP_RE.sub('', text)

In [2]:
df = pd.read_parquet('embedding_output/ci_flows_embedded.parquet')
embeddings = np.load('embedding_output/ci_flow_embeddings.npy')
print(f'{len(df)} flows, {embeddings.shape[1]}D embeddings')
print(f'Books: {df["book"].value_counts().to_dict()}')
n_clusters = df[df["cluster"] != -1]["cluster"].nunique()
n_noise = (df["cluster"] == -1).sum()
print(f'Clusters: {n_clusters}, Noise: {n_noise}')

# Flow text without appropriateness — used for NLI so the model judges
# the substance of the flow rather than the appropriateness label.
df['flow_text_nli'] = df['flow_text'].map(strip_appropriateness)
print(f'Example flow_text_nli: {df["flow_text_nli"].iloc[0][:120]}')

2988 flows, 384D embeddings
Books: {'Pride & Prejudice': 1754, '1984': 1234}
Clusters: 182, Noise: 589
Example flow_text_nli: In a state/political context, big brother (state) sends state propaganda (production of pig-iron), to public, via state 


## 1. UMAP scatter — colored by book

In [3]:
fig, ax = plt.subplots(figsize=(12, 8))
book_colors = {'1984': 'steelblue', 'Pride & Prejudice': 'salmon'}
for book, color in book_colors.items():
    mask = df['book'] == book
    ax.scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
              c=color, label=f'{book} (n={mask.sum()})', alpha=0.5, s=12, edgecolors='none')
ax.set_title('CI Flows in Shared Embedding Space (by book)', fontsize=14)
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.legend(fontsize=11)
plt.tight_layout()
savefig(fig, '01_umap_by_book.png')

**Plot saved:** [01_umap_by_book.png](embedding_output/plots/01_umap_by_book.png)

## 2. UMAP scatter — colored by cluster

In [4]:
fig, ax = plt.subplots(figsize=(12, 8))

noise = df[df['cluster'] == -1]
clustered = df[df['cluster'] != -1]

ax.scatter(noise['umap_x'], noise['umap_y'],
          c='lightgrey', alpha=0.25, s=8, label=f'noise (n={len(noise)})', edgecolors='none')
sc = ax.scatter(clustered['umap_x'], clustered['umap_y'],
               c=clustered['cluster'], cmap='tab20', alpha=0.6, s=12, edgecolors='none')
ax.set_title('CI Flows in Shared Embedding Space (by cluster)', fontsize=14)
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.legend(fontsize=11)
plt.tight_layout()
savefig(fig, '02_umap_by_cluster.png')

**Plot saved:** [02_umap_by_cluster.png](embedding_output/plots/02_umap_by_cluster.png)

## 3. UMAP scatter — colored by CI context

In [5]:
# color by the top N most common contexts, group the rest as 'other'
TOP_N_CONTEXTS = 8
top_contexts = df['ci_context'].value_counts().head(TOP_N_CONTEXTS).index.tolist()
df['ctx_display'] = df['ci_context'].where(df['ci_context'].isin(top_contexts), other='other')

palette = list(mcolors.TABLEAU_COLORS.values())[:TOP_N_CONTEXTS] + ['lightgrey']
ctx_order = top_contexts + ['other']

fig, ax = plt.subplots(figsize=(12, 8))
# plot 'other' first so it sits behind
for i, ctx in enumerate(reversed(ctx_order)):
    mask = df['ctx_display'] == ctx
    color = palette[ctx_order.index(ctx)]
    alpha = 0.2 if ctx == 'other' else 0.6
    size = 6 if ctx == 'other' else 14
    ax.scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
              c=color, label=f'{ctx} ({mask.sum()})', alpha=alpha, s=size, edgecolors='none')
ax.set_title('CI Flows by Context', fontsize=14)
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.legend(fontsize=9, loc='best', ncol=2)
plt.tight_layout()
savefig(fig, '03_umap_by_context.png')

**Plot saved:** [03_umap_by_context.png](embedding_output/plots/03_umap_by_context.png)

## 4. UMAP scatter — colored by transmission principle

In [6]:
TOP_N_TP = 8
top_tp = df['ci_transmission_principle'].value_counts().head(TOP_N_TP).index.tolist()
df['tp_display'] = df['ci_transmission_principle'].where(
    df['ci_transmission_principle'].isin(top_tp), other='other')

palette_tp = list(mcolors.TABLEAU_COLORS.values())[:TOP_N_TP] + ['lightgrey']
tp_order = top_tp + ['other']

fig, ax = plt.subplots(figsize=(12, 8))
for ctx in reversed(tp_order):
    mask = df['tp_display'] == ctx
    color = palette_tp[tp_order.index(ctx)]
    alpha = 0.2 if ctx == 'other' else 0.6
    size = 6 if ctx == 'other' else 14
    ax.scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
              c=color, label=f'{ctx} ({mask.sum()})', alpha=alpha, s=size, edgecolors='none')
ax.set_title('CI Flows by Transmission Principle', fontsize=14)
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.legend(fontsize=9, loc='best', ncol=2)
plt.tight_layout()
savefig(fig, '04_umap_by_transmission_principle.png')

**Plot saved:** [04_umap_by_transmission_principle.png](embedding_output/plots/04_umap_by_transmission_principle.png)

## 5. UMAP scatter — colored by appropriateness

In [7]:
fig, ax = plt.subplots(figsize=(12, 8))
approp_colors = {'appropriate': 'mediumseagreen', 'inappropriate': 'tomato'}
# plot any other values in grey first
other_mask = ~df['ci_appropriateness'].isin(approp_colors.keys())
if other_mask.any():
    ax.scatter(df.loc[other_mask, 'umap_x'], df.loc[other_mask, 'umap_y'],
              c='lightgrey', alpha=0.3, s=8, label=f'other ({other_mask.sum()})', edgecolors='none')
for approp, color in approp_colors.items():
    mask = df['ci_appropriateness'] == approp
    ax.scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
              c=color, label=f'{approp} ({mask.sum()})', alpha=0.5, s=12, edgecolors='none')
ax.set_title('CI Flows by Appropriateness', fontsize=14)
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.legend(fontsize=11)
plt.tight_layout()
savefig(fig, '05_umap_by_appropriateness.png')

**Plot saved:** [05_umap_by_appropriateness.png](embedding_output/plots/05_umap_by_appropriateness.png)

## 6. Book composition per cluster (stacked bar)

In [8]:
# compute per-cluster book fractions, sorted by 1984 fraction descending
clustered = df[df['cluster'] != -1].copy()
comp = clustered.groupby('cluster')['book'].value_counts().unstack(fill_value=0)
comp['total'] = comp.sum(axis=1)
comp['frac_1984'] = comp.get('1984', 0) / comp['total']
comp = comp.sort_values('frac_1984', ascending=False)

# only show clusters with at least 10 flows for readability
comp_show = comp[comp['total'] >= 10].copy()

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(comp_show))
bar_1984 = comp_show.get('1984', pd.Series(0, index=comp_show.index)).values
bar_pandp = comp_show.get('Pride & Prejudice', pd.Series(0, index=comp_show.index)).values

ax.bar(x, bar_1984, color='steelblue', label='1984')
ax.bar(x, bar_pandp, bottom=bar_1984, color='salmon', label='Pride & Prejudice')
ax.set_xticks(x)
ax.set_xticklabels([f'C{c}' for c in comp_show.index], rotation=90, fontsize=7)
ax.set_ylabel('Number of flows')
ax.set_title('Book composition per cluster (clusters with >= 10 flows)', fontsize=13)
ax.legend()
plt.tight_layout()
savefig(fig, '06_book_composition_per_cluster.png')

**Plot saved:** [06_book_composition_per_cluster.png](embedding_output/plots/06_book_composition_per_cluster.png)

## 7. Most book-distinctive clusters

In [9]:
# find clusters dominated by one book vs the other
clustered = df[df['cluster'] != -1].copy()
comp = clustered.groupby('cluster')['book'].value_counts().unstack(fill_value=0)
comp['total'] = comp.sum(axis=1)
comp['frac_1984'] = comp.get('1984', 0) / comp['total']

# only consider clusters with >= 8 flows
comp = comp[comp['total'] >= 8].copy()

print('=== Clusters dominated by 1984 ===')
top_1984 = comp.sort_values('frac_1984', ascending=False).head(10)
for cid, row in top_1984.iterrows():
    examples = clustered[clustered['cluster'] == cid]['flow_text'].head(2).tolist()
    print(f'\nCluster {cid}: {int(row["total"])} flows ({row["frac_1984"]:.0%} 1984)')
    for ex in examples:
        print(f'  - {ex[:130]}')

print('\n\n=== Clusters dominated by Pride & Prejudice ===')
top_pandp = comp.sort_values('frac_1984', ascending=True).head(10)
for cid, row in top_pandp.iterrows():
    examples = clustered[clustered['cluster'] == cid]['flow_text'].head(2).tolist()
    pct = 1 - row['frac_1984']
    print(f'\nCluster {cid}: {int(row["total"])} flows ({pct:.0%} P&P)')
    for ex in examples:
        print(f'  - {ex[:130]}')

=== Clusters dominated by 1984 ===

Cluster 0: 13 flows (100% 1984)
  - In a state/political context, narrator sends state secrets or classified information, to public, via state mandate. This flow is c
  - In a state/political context, narrator sends state secrets or classified information, to reader, via state mandate. This flow is c

Cluster 1: 9 flows (100% 1984)
  - In a state/political context, big brother (state) sends state propaganda (production of pig-iron), to public, via state mandate. T
  - In a state/political context, state (big brother) sends state secrets or classified information, to public, via surveillance. This

Cluster 4: 9 flows (100% 1984)
  - In a state/political context, ordinary party member sends state secrets or classified information, to public, via confidentiality.
  - In a state/political context, ordinary party member sends state secrets or classified information, about ordinary party member, to

Cluster 6: 13 flows (100% 1984)
  - In a state/political 

## 8. Cluster-level context heatmap

In [10]:
# build a cluster x context matrix (top 20 clusters by size, top 8 contexts)
clustered = df[df['cluster'] != -1].copy()
top_clusters = clustered['cluster'].value_counts().head(20).index.tolist()
top_contexts = clustered['ci_context'].value_counts().head(8).index.tolist()

sub = clustered[clustered['cluster'].isin(top_clusters)].copy()
sub['ctx_display'] = sub['ci_context'].where(sub['ci_context'].isin(top_contexts), other='other')

heat = sub.groupby('cluster')['ctx_display'].value_counts().unstack(fill_value=0)
# normalize each row to fractions
heat_norm = heat.div(heat.sum(axis=1), axis=0)
# reorder columns by overall frequency
col_order = [c for c in (top_contexts + ['other']) if c in heat_norm.columns]
heat_norm = heat_norm[col_order]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(heat_norm.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(heat_norm.columns)))
ax.set_xticklabels(heat_norm.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(heat_norm.index)))
ax.set_yticklabels([f'C{c} (n={heat.loc[c].sum()})' for c in heat_norm.index], fontsize=9)
ax.set_title('Context distribution within top-20 clusters', fontsize=13)
plt.colorbar(im, ax=ax, label='Fraction of cluster')
plt.tight_layout()
savefig(fig, '08_cluster_context_heatmap.png')

**Plot saved:** [08_cluster_context_heatmap.png](embedding_output/plots/08_cluster_context_heatmap.png)

## 9. Density contour — 1984 vs P&P

In [11]:
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)

for ax, (book, color) in zip(axes, [('1984', 'Blues'), ('Pride & Prejudice', 'Reds')]):
    sub = df[df['book'] == book]
    x, y = sub['umap_x'].values, sub['umap_y'].values
    
    # scatter
    ax.scatter(x, y, alpha=0.15, s=6, c='grey', edgecolors='none')
    
    # density contours
    try:
        xy = np.vstack([x, y])
        kde = gaussian_kde(xy)
        xmin, xmax = x.min() - 1, x.max() + 1
        ymin, ymax = y.min() - 1, y.max() + 1
        xx, yy = np.mgrid[xmin:xmax:200j, ymin:ymax:200j]
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contourf(xx, yy, zz, levels=10, cmap=color, alpha=0.5)
        ax.contour(xx, yy, zz, levels=10, colors='k', linewidths=0.3, alpha=0.4)
    except Exception as e:
        print(f'KDE failed for {book}: {e}')
    
    ax.set_title(f'{book} (n={len(sub)})', fontsize=13)
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')

plt.suptitle('Density of CI flows in embedding space', fontsize=14, y=1.01)
plt.tight_layout()
savefig(fig, '09_density_contour.png')

**Plot saved:** [09_density_contour.png](embedding_output/plots/09_density_contour.png)

## 10. Shared vs book-specific regions

In [12]:
# for each cluster, classify as 1984-dominant, P&P-dominant, or shared
clustered = df[df['cluster'] != -1].copy()
comp = clustered.groupby('cluster')['book'].value_counts().unstack(fill_value=0)
comp['total'] = comp.sum(axis=1)
comp['frac_1984'] = comp.get('1984', 0) / comp['total']

def classify_cluster(row):
    if row['frac_1984'] >= 0.8:
        return '1984-dominant'
    elif row['frac_1984'] <= 0.2:
        return 'P&P-dominant'
    else:
        return 'shared'

comp['dominance'] = comp.apply(classify_cluster, axis=1)
cluster_dom = comp['dominance'].to_dict()

clustered['dominance'] = clustered['cluster'].map(cluster_dom)

dom_colors = {'1984-dominant': 'steelblue', 'P&P-dominant': 'salmon', 'shared': 'mediumpurple'}

fig, ax = plt.subplots(figsize=(12, 8))
# noise in background
noise = df[df['cluster'] == -1]
ax.scatter(noise['umap_x'], noise['umap_y'],
          c='lightgrey', alpha=0.15, s=6, label=f'noise ({len(noise)})', edgecolors='none')

for dom in ['shared', '1984-dominant', 'P&P-dominant']:
    mask = clustered['dominance'] == dom
    ax.scatter(clustered.loc[mask, 'umap_x'], clustered.loc[mask, 'umap_y'],
              c=dom_colors[dom], label=f'{dom} ({mask.sum()})', alpha=0.5, s=12, edgecolors='none')

ax.set_title('Shared vs Book-Specific Flow Regions', fontsize=14)
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.legend(fontsize=11)
plt.tight_layout()
savefig(fig, '10_shared_vs_book_specific.png')

# summary stats
print('\nCluster dominance summary:')
for dom in ['1984-dominant', 'P&P-dominant', 'shared']:
    n_clusters = (comp['dominance'] == dom).sum()
    n_flows = clustered[clustered['dominance'] == dom].shape[0]
    print(f'  {dom}: {n_clusters} clusters, {n_flows} flows')

**Plot saved:** [10_shared_vs_book_specific.png](embedding_output/plots/10_shared_vs_book_specific.png)


Cluster dominance summary:
  1984-dominant: 78 clusters, 944 flows
  P&P-dominant: 101 clusters, 1422 flows
  shared: 3 clusters, 33 flows


## 11. Nearest cross-book neighbors (which flows are most similar across books?)

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

mask_1984 = (df['book'] == '1984').values
mask_pandp = (df['book'] == 'Pride & Prejudice').values

emb_1984 = embeddings[mask_1984]
emb_pandp = embeddings[mask_pandp]

# compute cross-book cosine similarity
sim = cosine_similarity(emb_1984, emb_pandp)  # (n_1984, n_pandp)

# for each 1984 flow, find its nearest P&P neighbor
best_pandp_idx = sim.argmax(axis=1)
best_pandp_sim = sim.max(axis=1)

df_1984 = df[mask_1984].reset_index(drop=True)
df_pandp = df[mask_pandp].reset_index(drop=True)

# show the top 15 most similar cross-book pairs
top_pairs_idx = np.argsort(-best_pandp_sim)[:15]

print('Top 15 most similar cross-book flow pairs:\n')
for rank, i in enumerate(top_pairs_idx):
    j = best_pandp_idx[i]
    print(f'{rank+1}. Similarity: {best_pandp_sim[i]:.3f}')
    print(f'   1984:  {df_1984.loc[i, "flow_text"][:120]}')
    print(f'   P&P:   {df_pandp.loc[j, "flow_text"][:120]}')
    print()

Top 15 most similar cross-book flow pairs:

1. Similarity: 0.927
   1984:  In a commerce context, community members sends social availability (arrivals, departures), to public, via social obligat
   P&P:   In a social etiquette context, community members sends social availability (arrivals, departures), to public, via social

2. Similarity: 0.916
   1984:  In a social etiquette context, first character sends personal feelings or sentiments, to second character, via reciproci
   P&P:   In a social etiquette context, character sends personal feelings or sentiments, about character, to listener, via recipr

3. Similarity: 0.904
   1984:  In a courtship context, woman sends personal feelings or sentiments, to man, via reciprocity. This flow is considered ap
   P&P:   In a courtship context, jane sends personal feelings or sentiments, about jane, to speaker, via reciprocity. This flow i

4. Similarity: 0.904
   1984:  In a courtship context, woman sends personal feelings or sentiments, to m

## 12. Cross-book similarity distribution

In [14]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(best_pandp_sim, bins=50, color='mediumpurple', alpha=0.7, edgecolor='white')
ax.axvline(np.median(best_pandp_sim), color='black', linestyle='--', label=f'median={np.median(best_pandp_sim):.3f}')
ax.set_xlabel('Cosine similarity to nearest P&P flow')
ax.set_ylabel('Count (1984 flows)')
ax.set_title('For each 1984 flow: similarity to its nearest Pride & Prejudice neighbor', fontsize=12)
ax.legend()
plt.tight_layout()
savefig(fig, '12_cross_book_similarity_dist.png')

**Plot saved:** [12_cross_book_similarity_dist.png](embedding_output/plots/12_cross_book_similarity_dist.png)

---
# NLI-Based Semantic Analysis of Cross-Book Flow Pairs

Using `cross-encoder/nli-deberta-v3-large` to classify the logical relationship (entailment / neutral / contradiction) between nearest-neighbor CI flow pairs across the two books.

In [15]:
from sentence_transformers import CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
import scipy.special

NLI_CACHE_PATH = Path('embedding_output/nli_cross_book_scores.parquet')
NLI_LABELS = ['contradiction', 'entailment', 'neutral']

# --- Reconstruct cross-book nearest neighbors ---
mask_1984 = (df['book'] == '1984').values
mask_pandp = (df['book'] == 'Pride & Prejudice').values

emb_1984 = embeddings[mask_1984]
emb_pandp = embeddings[mask_pandp]

df_1984 = df[mask_1984].reset_index(drop=True)
df_pandp = df[mask_pandp].reset_index(drop=True)

sim = cosine_similarity(emb_1984, emb_pandp)
best_pandp_idx = sim.argmax(axis=1)
best_pandp_sim = sim.max(axis=1)

# --- Compute NLI scores (or load from cache) ---
if NLI_CACHE_PATH.exists():
    print(f'Loading cached NLI scores from {NLI_CACHE_PATH}')
    nli_df = pd.read_parquet(NLI_CACHE_PATH)
else:
    print('Loading NLI model: cross-encoder/nli-deberta-v3-large ...')
    nli_model = CrossEncoder('cross-encoder/nli-deberta-v3-large')
    
    # Build pairs: each 1984 flow text -> its nearest P&P flow text
    # Use flow_text_nli (without appropriateness) so NLI judges the flow substance
    pairs = []
    for i in range(len(df_1984)):
        j = best_pandp_idx[i]
        pairs.append((df_1984.loc[i, 'flow_text_nli'], df_pandp.loc[j, 'flow_text_nli']))
    
    print(f'Running NLI on {len(pairs)} cross-book pairs ...')
    raw_scores = nli_model.predict(pairs, show_progress_bar=True)
    probs = scipy.special.softmax(raw_scores, axis=1)  # (N, 3)
    
    nli_df = pd.DataFrame({
        'idx_1984': range(len(df_1984)),
        'idx_pandp': best_pandp_idx,
        'cosine_sim': best_pandp_sim,
        'flow_text_1984': [p[0] for p in pairs],
        'flow_text_pandp': [p[1] for p in pairs],
        'ci_context_1984': df_1984['ci_context'].values,
        'ci_context_pandp': df_pandp.iloc[best_pandp_idx].reset_index(drop=True)['ci_context'].values,
        'score_contradiction': probs[:, 0],
        'score_entailment': probs[:, 1],
        'score_neutral': probs[:, 2],
        'nli_label': [NLI_LABELS[i] for i in probs.argmax(axis=1)],
    })
    
    nli_df.to_parquet(NLI_CACHE_PATH, index=False)
    print(f'Cached NLI scores to {NLI_CACHE_PATH}')
    del nli_model  # free memory

print(f'NLI pairs: {len(nli_df)}')
print(f'Label distribution:\n{nli_df["nli_label"].value_counts()}')

/share/pierson/matt/UAIR/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Loading NLI model: cross-encoder/nli-deberta-v3-large ...
Running NLI on 1234 cross-book pairs ...


Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Cached NLI scores to embedding_output/nli_cross_book_scores.parquet
NLI pairs: 1234
Label distribution:
nli_label
contradiction    1111
neutral           109
entailment         14
Name: count, dtype: int64


## 13. Analysis A: Cross-book NLI label distribution

In [16]:
nli_colors = {'entailment': 'mediumseagreen', 'neutral': 'silver', 'contradiction': 'tomato'}
label_counts = nli_df['nli_label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# bar chart
ax = axes[0]
for label in NLI_LABELS:
    count = label_counts.get(label, 0)
    ax.bar(label, count, color=nli_colors[label])
    ax.text(label, count + 10, str(count), ha='center', fontsize=11)
ax.set_ylabel('Number of cross-book pairs')
ax.set_title('NLI classification of nearest cross-book flow pairs')

# pie chart
ax = axes[1]
sizes = [label_counts.get(l, 0) for l in NLI_LABELS]
colors = [nli_colors[l] for l in NLI_LABELS]
ax.pie(sizes, labels=NLI_LABELS, colors=colors, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
ax.set_title('NLI label proportions')

plt.tight_layout()
savefig(fig, '13_nli_label_distribution.png')

**Plot saved:** [13_nli_label_distribution.png](embedding_output/plots/13_nli_label_distribution.png)

## 14. Analysis B: UMAP scatter — 1984 flows colored by NLI relation to nearest P&P neighbor

In [17]:
fig, ax = plt.subplots(figsize=(12, 8))

# P&P points as background
ax.scatter(df.loc[mask_pandp, 'umap_x'], df.loc[mask_pandp, 'umap_y'],
          c='whitesmoke', alpha=0.3, s=8, label='P&P (background)', edgecolors='none')

# 1984 points colored by NLI label to nearest P&P neighbor
umap_1984 = df.loc[mask_1984, ['umap_x', 'umap_y']].reset_index(drop=True)
for label in ['neutral', 'entailment', 'contradiction']:  # plot contradiction on top
    mask = nli_df['nli_label'] == label
    ax.scatter(umap_1984.loc[mask, 'umap_x'], umap_1984.loc[mask, 'umap_y'],
              c=nli_colors[label], label=f'{label} ({mask.sum()})',
              alpha=0.6, s=16, edgecolors='none')

ax.set_title('1984 flows colored by NLI relation to nearest P&P neighbor', fontsize=13)
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.legend(fontsize=10)
plt.tight_layout()
savefig(fig, '14_umap_nli_relation.png')

**Plot saved:** [14_umap_nli_relation.png](embedding_output/plots/14_umap_nli_relation.png)

## 15. Analysis C: Contradiction mining — most normatively opposed cross-book pairs

In [18]:
top_contradictions = nli_df.sort_values('score_contradiction', ascending=False).head(20)

print('Top 20 cross-book contradictions (flows with opposing normative patterns):\n')
for rank, (_, row) in enumerate(top_contradictions.iterrows()):
    print(f'{rank+1}. Contradiction score: {row["score_contradiction"]:.3f}  |  Cosine sim: {row["cosine_sim"]:.3f}')
    print(f'   1984 [{row["ci_context_1984"]}]:')
    print(f'     {row["flow_text_1984"][:160]}')
    print(f'   P&P  [{row["ci_context_pandp"]}]:')
    print(f'     {row["flow_text_pandp"][:160]}')
    print()

Top 20 cross-book contradictions (flows with opposing normative patterns):

1. Contradiction score: 1.000  |  Cosine sim: 0.597
   1984 [state/political]:
     In a state/political context, winston sends misconduct or moral failings, about aaronson, rutherford, and jones, to winston, via confidentiality.
   P&P  [family]:
     In a family context, elizabeth sends misconduct or moral failings, about lydia, to darcy, via confidentiality.

2. Contradiction score: 1.000  |  Cosine sim: 0.611
   1984 [forbidden relationship]:
     In a forbidden relationship context, julia sends misconduct or moral failings, about julia, to winston, via consent.
   P&P  [courtship]:
     In a courtship context, mr. darcy sends misconduct or moral failings, about jane, to elizabeth, via confidentiality.

3. Contradiction score: 1.000  |  Cosine sim: 0.533
   1984 [state/political]:
     In a state/political context, winston sends personal history or past actions, about aaronson, rutherford, and jones, to win

## 16. Analysis D: NLI label distribution by CI context

In [19]:
# group by 1984's ci_context, show NLI label fractions
top_ctx = nli_df['ci_context_1984'].value_counts().head(10).index.tolist()
ctx_nli = nli_df[nli_df['ci_context_1984'].isin(top_ctx)].copy()

# compute fractions per context
ctx_label_counts = ctx_nli.groupby('ci_context_1984')['nli_label'].value_counts().unstack(fill_value=0)
ctx_label_frac = ctx_label_counts.div(ctx_label_counts.sum(axis=1), axis=0)

# order by contradiction fraction descending
ctx_label_frac = ctx_label_frac.sort_values('contradiction', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(ctx_label_frac))
left = np.zeros(len(ctx_label_frac))

for label in ['entailment', 'neutral', 'contradiction']:
    vals = ctx_label_frac.get(label, pd.Series(0, index=ctx_label_frac.index)).values
    ax.barh(y_pos, vals, left=left, color=nli_colors[label], label=label)
    left += vals

ax.set_yticks(y_pos)
sizes = ctx_label_counts.sum(axis=1).reindex(ctx_label_frac.index)
ax.set_yticklabels([f'{ctx} (n={int(sizes[ctx])})' for ctx in ctx_label_frac.index])
ax.set_xlabel('Fraction of cross-book pairs')
ax.set_title('NLI label distribution by CI context (1984 flows)', fontsize=13)
ax.legend(loc='lower right')
ax.set_xlim(0, 1)
plt.tight_layout()
savefig(fig, '16_nli_by_ci_context.png')

**Plot saved:** [16_nli_by_ci_context.png](embedding_output/plots/16_nli_by_ci_context.png)

## 17. Analysis E: Within-book vs cross-book NLI comparison

In [20]:
WITHIN_CACHE_PATH = Path('embedding_output/nli_within_book_scores.parquet')

if WITHIN_CACHE_PATH.exists():
    print(f'Loading cached within-book NLI scores from {WITHIN_CACHE_PATH}')
    within_nli_df = pd.read_parquet(WITHIN_CACHE_PATH)
else:
    print('Loading NLI model for within-book comparison ...')
    nli_model = CrossEncoder('cross-encoder/nli-deberta-v3-large')
    
    within_pairs = []
    within_meta = []
    
    np.random.seed(42)
    N_SAMPLE = 250  # per book
    
    for book_name, book_emb, book_df_sub in [
        ('1984', emb_1984, df_1984),
        ('Pride & Prejudice', emb_pandp, df_pandp),
    ]:
        # compute within-book similarity
        within_sim = cosine_similarity(book_emb)
        np.fill_diagonal(within_sim, -1)  # exclude self
        
        # sample N random flows, take their nearest within-book neighbor
        sample_idx = np.random.choice(len(book_df_sub), size=min(N_SAMPLE, len(book_df_sub)), replace=False)
        for i in sample_idx:
            j = within_sim[i].argmax()
            within_pairs.append((book_df_sub.loc[i, 'flow_text_nli'], book_df_sub.loc[j, 'flow_text_nli']))
            within_meta.append({
                'book': book_name,
                'idx_a': i, 'idx_b': j,
                'cosine_sim': within_sim[i, j],
            })
    
    print(f'Running NLI on {len(within_pairs)} within-book pairs ...')
    raw_scores = nli_model.predict(within_pairs, show_progress_bar=True)
    probs = scipy.special.softmax(raw_scores, axis=1)
    
    within_nli_df = pd.DataFrame(within_meta)
    within_nli_df['score_contradiction'] = probs[:, 0]
    within_nli_df['score_entailment'] = probs[:, 1]
    within_nli_df['score_neutral'] = probs[:, 2]
    within_nli_df['nli_label'] = [NLI_LABELS[i] for i in probs.argmax(axis=1)]
    
    within_nli_df.to_parquet(WITHIN_CACHE_PATH, index=False)
    print(f'Cached within-book NLI scores to {WITHIN_CACHE_PATH}')
    del nli_model

# --- Compare within-book vs cross-book ---
cross_counts = nli_df['nli_label'].value_counts()
within_counts = within_nli_df['nli_label'].value_counts()

cross_frac = cross_counts / cross_counts.sum()
within_frac = within_counts / within_counts.sum()

comparison = pd.DataFrame({
    'Cross-book': [cross_frac.get(l, 0) for l in NLI_LABELS],
    'Within-book': [within_frac.get(l, 0) for l in NLI_LABELS],
}, index=NLI_LABELS)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(NLI_LABELS))
width = 0.35

bars1 = ax.bar(x - width/2, comparison['Cross-book'], width,
               color=[nli_colors[l] for l in NLI_LABELS], edgecolor='black', linewidth=0.8, label='Cross-book')
bars2 = ax.bar(x + width/2, comparison['Within-book'], width,
               color=[nli_colors[l] for l in NLI_LABELS], edgecolor='black', linewidth=0.8,
               alpha=0.5, hatch='///', label='Within-book')

ax.set_xticks(x)
ax.set_xticklabels(NLI_LABELS, fontsize=11)
ax.set_ylabel('Fraction of pairs')
ax.set_title('NLI label distribution: cross-book vs within-book nearest neighbors', fontsize=12)
ax.legend(fontsize=10)

# add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=9)

plt.tight_layout()
savefig(fig, '17_within_vs_cross_book_nli.png')

# per-book within breakdown
print('\nWithin-book NLI breakdown:')
for book in ['1984', 'Pride & Prejudice']:
    sub = within_nli_df[within_nli_df['book'] == book]
    counts = sub['nli_label'].value_counts()
    print(f'  {book}: {dict(counts)}')

Loading NLI model for within-book comparison ...
Running NLI on 500 within-book pairs ...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Cached within-book NLI scores to embedding_output/nli_within_book_scores.parquet


**Plot saved:** [17_within_vs_cross_book_nli.png](embedding_output/plots/17_within_vs_cross_book_nli.png)


Within-book NLI breakdown:
  1984: {'contradiction': np.int64(108), 'entailment': np.int64(75), 'neutral': np.int64(67)}
  Pride & Prejudice: {'contradiction': np.int64(129), 'neutral': np.int64(63), 'entailment': np.int64(58)}


## 18. Analysis F: Cosine similarity vs entailment probability

In [21]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Left: cosine sim vs entailment prob, colored by NLI label ---
ax = axes[0]
for label in ['neutral', 'entailment', 'contradiction']:
    mask = nli_df['nli_label'] == label
    ax.scatter(nli_df.loc[mask, 'cosine_sim'], nli_df.loc[mask, 'score_entailment'],
              c=nli_colors[label], label=label, alpha=0.4, s=12, edgecolors='none')
ax.set_xlabel('Cosine similarity (embedding space)')
ax.set_ylabel('Entailment probability (NLI)')
ax.set_title('Embedding similarity vs NLI entailment', fontsize=12)
ax.legend(fontsize=10)

# --- Right: cosine sim vs contradiction prob, colored by NLI label ---
ax = axes[1]
for label in ['neutral', 'entailment', 'contradiction']:
    mask = nli_df['nli_label'] == label
    ax.scatter(nli_df.loc[mask, 'cosine_sim'], nli_df.loc[mask, 'score_contradiction'],
              c=nli_colors[label], label=label, alpha=0.4, s=12, edgecolors='none')
ax.set_xlabel('Cosine similarity (embedding space)')
ax.set_ylabel('Contradiction probability (NLI)')
ax.set_title('Embedding similarity vs NLI contradiction', fontsize=12)
ax.legend(fontsize=10)

plt.suptitle('Does embedding proximity imply normative alignment?', fontsize=14, y=1.02)
plt.tight_layout()
savefig(fig, '18_cosine_sim_vs_nli.png')

# highlight the most interesting quadrant: high cosine sim + high contradiction
high_sim_contradictions = nli_df[
    (nli_df['cosine_sim'] >= nli_df['cosine_sim'].quantile(0.75)) & 
    (nli_df['nli_label'] == 'contradiction')
].sort_values('score_contradiction', ascending=False)

print(f'\nHigh-similarity contradictions ({len(high_sim_contradictions)} pairs where cosine >= 75th pctile but NLI says contradiction):\n')
for _, row in high_sim_contradictions.head(10).iterrows():
    print(f'  Cosine: {row["cosine_sim"]:.3f} | Contradiction: {row["score_contradiction"]:.3f}')
    print(f'    1984: {row["flow_text_1984"][:120]}')
    print(f'    P&P:  {row["flow_text_pandp"][:120]}')
    print()

**Plot saved:** [18_cosine_sim_vs_nli.png](embedding_output/plots/18_cosine_sim_vs_nli.png)


High-similarity contradictions (263 pairs where cosine >= 75th pctile but NLI says contradiction):

  Cosine: 0.695 | Contradiction: 1.000
    1984: In a family context, parsons sends misconduct or moral failings, about parsons' son, to winston, via social obligation.
    P&P:  In a family context, mr. collins sends misconduct or moral failings, about lydia bennet, to mr. bennet, via advice.

  Cosine: 0.778 | Contradiction: 1.000
    1984: In a courtship context, julia sends personal feelings or sentiments, about julia, to o'brien, via consent.
    P&P:  In a courtship context, colonel forster sends personal feelings or sentiments, about lydia, to jane, via consent.

  Cosine: 0.726 | Contradiction: 1.000
    1984: In a family context, his mother sends financial standing or income (in the form of food portions), about boy, to boy, vi
    P&P:  In a family context, mr. bennet sends financial standing or income, about mr. bennet's daughter, to mrs. bennet, via ent

  Cosine: 0.747 | Co